In [7]:
%pip install -q numpy scipy pandas matplotlib seaborn scikit-learn torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [9]:
import scipy
import numpy as np # manipulate N-dimensional arrays
import pandas as pd # data frame
import matplotlib.pyplot as plt # data plotting
import seaborn # advanced data plotting
from sklearn import preprocessing # basic ML models

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

In [10]:
device = "cuda:0" if torch.cuda.is_available() else "cpu" #0 indicates the GPU you gonna use
print(f"Using {device} device")

Using cpu device


In [11]:
from torchvision import datasets
from torchvision import transforms

transform = transforms.Compose([   # kết hợp nhiều biến đổi trong 1 quy trình 
    transforms.ToTensor(),  # chuyển ảnh thành tensor
    transforms.Normalize((0.5,), (0.5,))   # chuẩn hoá giá trị pixel  với mean = 0.5, std = 0.5 và như thế x mới sẽ có dạng (x - mean)/std
])

traindata = datasets.MNIST(
    root="~/data",train=True,download=True,
    transform=transform
)

testdata = datasets.MNIST(
    root="~/data",train=False,download=True,
    transform=transform
)


100%|██████████| 9.91M/9.91M [00:00<00:00, 60.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.68MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.29MB/s]


In [12]:
traindata

Dataset MNIST
    Number of datapoints: 60000
    Root location: /root/data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [14]:
import numpy as np
from torch.utils.data import Subset, DataLoader

# 1. Lấy mảng nhãn từ traindata hiện tại của bạn
targets = np.array(traindata.targets)
labeled_indices = []

# 2. Chọn ngẫu nhiên 10 mẫu cho mỗi chữ số từ 0 đến 9 (tổng cộng 100 mẫu)
num_labeled_per_class = 10
np.random.seed(42) # Đặt seed để kết quả đồng nhất mỗi lần chạy

for digit in range(10):
    indices_of_digit = np.where(targets == digit)[0]
    selected = np.random.choice(indices_of_digit, num_labeled_per_class, replace=False)
    labeled_indices.extend(selected)

# 3. Phần còn lại của tập train chính là tập không nhãn (Unlabeled)
all_indices = np.arange(len(traindata))
unlabeled_indices = np.setdiff1d(all_indices, labeled_indices)

# 4. Tạo các Subset từ traindata sẵn có
labeled_subset = Subset(traindata, labeled_indices)
unlabeled_subset = Subset(traindata, unlabeled_indices)

# 5. Đóng gói vào DataLoader để chuẩn bị đưa vào mô hình
batch_size = 64
labeled_loader = DataLoader(labeled_subset, batch_size=batch_size, shuffle=True)
unlabeled_loader = DataLoader(unlabeled_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(testdata, batch_size=batch_size, shuffle=False)

print(f"Phân tách dữ liệu thành công!")
print(f"- Số mẫu có nhãn (Fine-tuning): {len(labeled_subset)}")
print(f"- Số mẫu không nhãn (Self-Supervised Pre-training): {len(unlabeled_subset)}")
print(f"- Số mẫu Test: {len(testdata)}")

Phân tách dữ liệu thành công!
- Số mẫu có nhãn (Fine-tuning): 100
- Số mẫu không nhãn (Self-Supervised Pre-training): 59900
- Số mẫu Test: 10000


/tmp/ipykernel_50/2519852593.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  targets = np.array(traindata.targets)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNEncoder(nn.Module):
    def __init__(self, feature_dim=128):
        super(CNNEncoder, self).__init__()
        
        # Phần trích xuất đặc trưng (Backbone CNN)
        # Input: [Batch_size, 1, 28, 28]
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1) # -> [B, 32, 28, 28]
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1) # -> [B, 64, 28, 28]
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)                  # Giảm nửa kích thước: 28x28 -> 14x14 -> 7x7
        
        # Phần Linear nén đặc trưng thành vector có số chiều = feature_dim (mặc định 128)
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, feature_dim)

    def forward(self, x):
        # Khối tích chập 1
        x = self.pool(F.relu(self.conv1(x))) # Kích thước còn 14x14
        # Khối tích chập 2
        x = self.pool(F.relu(self.conv2(x))) # Kích thước còn 7x7
        
        # Trải phẳng để đưa vào lớp Fully Connected
        x = x.view(x.size(0), -1) 
        x = F.relu(self.fc1(x))
        
        # Lớp chiếu cuối cùng để tạo vector đặc trưng z
        z = self.fc2(x)
        
        # Chuẩn hóa L2 (L2-normalization) cực kỳ quan trọng trong SSL để đưa vector về độ dài bằng 1
        z = F.normalize(z, p=2, dim=1)
        return z

# Khởi tạo thử nghiệm mạng f và kiểm tra với 1 batch ảnh bất kỳ từ unlabeled_loader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
f_encoder = CNNEncoder(feature_dim=128).to(device)

print(f"Đã khởi tạo xong mạng Encoder f trên thiết bị: {device}")
print(f_encoder)

# Test thử chạy 1 batch qua mạng xem vector đầu ra có đúng hình dạng không
sample_images, _ = next(iter(unlabeled_loader))
sample_images = sample_images.to(device)
sample_z = f_encoder(sample_images)
print(f"Kích thước vector đặc trưng đầu ra (Batch_size, Feature_dim): {sample_z.shape}")

Đã khởi tạo xong mạng Encoder f trên thiết bị: cpu
CNNEncoder(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
)


NameError: name 'unlabeled_loader' is not defined

In [16]:
import copy

# 1. Tạo mạng g có kiến trúc y hệt f và đẩy lên thiết bị (GPU/CPU)
g_encoder = CNNEncoder(feature_dim=128).to(device)

# 2. Sao chép nguyên bản toàn bộ trọng số (weights) từ f sang g ở thời điểm ban đầu
g_encoder.load_state_dict(f_encoder.state_dict())

# 3. Tắt hoàn toàn việc tính toán gradient cho mạng g (vì g không học qua backpropagation)
for param in g_encoder.parameters():
    param.requires_grad = False

print("Đã khởi tạo xong mạng g (Target Encoder) thành công!")

# 4. Viết hàm cập nhật trọng số cho g theo công thức EMA sau mỗi bước học
def update_momentum_encoder(f_net, g_net, m=0.999):
    """
    Cập nhật trọng số của g dựa trên f:
    theta_g = m * theta_g + (1 - m) * theta_f
    (m gần bằng 1, ví dụ 0.999, giúp g thay đổi rất từ từ và ổn định)
    """
    with torch.no_grad():
        for param_f, param_g in zip(f_net.parameters(), g_net.parameters()):
            param_g.data.mul_(m).add_(param_f.data, alpha=1 - m)

print("Đã định nghĩa xong hàm cập nhật EMA cho g!")

Đã khởi tạo xong mạng g (Target Encoder) thành công!
Đã định nghĩa xong hàm cập nhật EMA cho g!


In [17]:
class MemoryBank(nn.Module):
    def __init__(self, feature_dim=128, memory_size=4096):
        super(MemoryBank, self).__init__()
        self.feature_dim = feature_dim
        self.memory_size = memory_size
        
        # Khởi tạo một ngân hàng bộ nhớ dưới dạng buffer của PyTorch (không tính là trọng số học tập qua backprop)
        # Ban đầu điền các giá trị ngẫu nhiên và chuẩn hóa L2
        init_memory = torch.randn(memory_size, feature_dim)
        init_memory = F.normalize(init_memory, dim=1)
        self.register_buffer("memory", init_memory)
        
        # Con trỏ để biết vị trí cần ghi đè tiếp theo trong hàng đợi
        self.register_buffer("ptr", torch.zeros(1, dtype=torch.long))

    @torch.no_grad()
    def update(self, z_keys):
        """
        Cập nhật kho bộ nhớ bằng các vector đặc trưng mới (z_keys từ mạng g)
        theo cơ chế cuốn chiếu (FIFO - First In, First Out).
        """
        batch_size = z_keys.shape[0]
        ptr = int(self.ptr)
        
        # Nếu số lượng batch vượt quá kích thước còn lại của bộ nhớ, ta cắt bớt
        if ptr + batch_size > self.memory_size:
            # Chỉ lấy phần vừa đủ để lấp đầy đến cuối kho
            batch_size = self.memory_size - ptr
            z_keys = z_keys[:batch_size]
        
        # Ghi đè vector mới vào vị trí con trỏ hiện tại
        self.memory[ptr:ptr + batch_size, :] = z_keys.detach()
        
        # Di chuyển con trỏ, nếu kịch kim thì quay lại từ đầu (vòng tròn)
        ptr = (ptr + batch_size) % self.memory_size
        self.ptr[0] = ptr

    def forward(self):
        # Trả về toàn bộ kho bộ nhớ hiện tại
        return self.memory

# Khởi tạo kho bộ nhớ M (ví dụ chứa 4096 vector, mỗi vector 128 chiều)
memory_bank = MemoryBank(feature_dim=128, memory_size=4096).to(device)

print(f"Đã khởi tạo xong Ngân hàng bộ nhớ M!")
print(f"Kích thước kho M (Số lượng mẫu, Số chiều): {memory_bank.memory.shape}")

Đã khởi tạo xong Ngân hàng bộ nhớ M!
Kích thước kho M (Số lượng mẫu, Số chiều): torch.Size([4096, 128])


In [18]:
import torch.optim as optim

def contrastive_loss(z1, z2, memory, temperature=0.07):
    """
    Hàm tính InfoNCE Loss dựa trên kho bộ nhớ M
    """
    # z1, z2 có kích thước [Batch_size, 128]
    # Tính độ tương đồng (cosine similarity) giữa z1 và z2 (dương tính)
    l_pos = torch.sum(z1 * z2, dim=-1, keepdim=True) # [B, 1]
    
    # Tính độ tương đồng giữa z1 và toàn bộ kho bộ nhớ M (âm tính)
    # memory có kích thước [Memory_size, 128]
    l_neg = torch.mm(z1, memory.t()) # [B, Memory_size]
    
    # Ghép lại thành logits và chia cho nhiệt độ (temperature scaling)
    logits = torch.cat([l_pos, l_neg], dim=1) / temperature
    
    # Nhãn chuẩn cho biết z1 khớp với vị trí dương tính đầu tiên (index 0)
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)
    
    return F.cross_entropy(logits, labels)

# Khởi tạo Optimizer (thuật toán tối ưu Adam cho f)
optimizer = optim.Adam(f_encoder.parameters(), lr=1e-3)

# Huấn luyện thử trong một số epoch (ví dụ: 5 epochs để kiểm tra mô hình chạy mượt không)
num_epochs = 5
print("Bắt đầu quá trình Self-Supervised Pre-training...")

for epoch in range(num_epochs):
    f_encoder.train()
    total_loss = 0.0
    
    for batch_idx, (images, _) in enumerate(unlabeled_loader):
        images = images.to(device)
        
        # 1. Tạo 2 góc nhìn ngẫu nhiên từ ảnh (Augmentation đơn giản bằng cách cộng chút nhiễu hoặc biến đổi nhẹ)
        # Ở đây ta tạo x1 và x2 bằng cách thêm nhiễu Gauss nhẹ để tạo sự khác biệt góc nhìn
        x1 = images + 0.1 * torch.randn_like(images)
        x2 = images + 0.1 * torch.randn_like(images)
        
        # Kẹp giá trị về đúng khoảng chuẩn hóa [-1, 1]
        x1 = torch.clamp(x1, -1.0, 1.0)
        x2 = torch.clamp(x2, -1.0, 1.0)
        
        # 2. Forward qua 2 mạng
        z1 = f_encoder(x1) # Online encoder
        
        with torch.no_grad():
            g_encoder.eval()
            z2 = g_encoder(x2) # Momentum encoder
            
        # 3. Tính Loss
        loss = contrastive_loss(z1, z2, memory_bank())
        
        # 4. Backpropagation & Cập nhật f
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 5. Cập nhật g theo EMA và cập nhật kho bộ nhớ M
        update_momentum_encoder(f_encoder, g_encoder, m=0.999)
        memory_bank.update(z2.detach())
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(unlabeled_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss tự giám sát: {avg_loss:.4f}")

print("Hoàn tất quá trình Pre-training tự giám sát!")

Bắt đầu quá trình Self-Supervised Pre-training...
Epoch [1/5] - Loss tự giám sát: 4.0424
Epoch [2/5] - Loss tự giám sát: 0.6615
Epoch [3/5] - Loss tự giám sát: 0.1065
Epoch [4/5] - Loss tự giám sát: 0.0520
Epoch [5/5] - Loss tự giám sát: 0.0392
Hoàn tất quá trình Pre-training tự giám sát!


In [2]:
import torch.nn as nn
import torch.optim as optim

# 1. Đóng băng hoàn toàn mạng Encoder f (không cập nhật trọng số nữa)
for param in f_encoder.parameters():
    param.requires_grad = False

# 2. Xây dựng một lớp Linear Classifier đơn giản (đầu vào 128 chiều từ f, đầu ra 10 lớp chữ số 0-9)
class LinearClassifier(nn.Module):
    def __init__(self, feature_dim=128, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(feature_dim, num_classes)
        
    def forward(self, z):
        return self.fc(z)

classifier = LinearClassifier(feature_dim=128, num_classes=10).to(device)

# 3. Cài đặt optimizer chỉ cho Linear Classifier và hàm Loss
optimizer_cls = optim.Adam(classifier.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 4. Huấn luyện Linear Classifier bằng đúng 100 mẫu có nhãn (labeled_loader)
num_finetune_epochs = 30
f_encoder.eval() # Đưa f về chế độ inference (chỉ trích xuất đặc trưng)

print("Bắt đầu fine-tune Linear Classifier với 100 mẫu có nhãn...")
for epoch in range(num_finetune_epochs):
    classifier.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in labeled_loader:
        images, labels = images.to(device), labels.to(device)
        
        with torch.no_grad():
            features = f_encoder(images) # Lấy vector 128 chiều từ f đã pre-train
            
        outputs = classifier(features)
        loss = criterion(outputs, labels)
        
        optimizer_cls.zero_grad()
        loss.backward()
        optimizer_cls.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    if (epoch + 1) % 10 == 0 or epoch == 0:
        acc = 100. * correct / total
        print(f"Epoch [{epoch+1}/{num_finetune_epochs}] - Loss: {total_loss/len(labeled_loader):.4f} - Acc trên 100 mẫu train: {acc:.2f}%")

# 5. Đánh giá toàn bộ mô hình trên tập Test chuẩn (10.000 ảnh)
classifier.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        features = f_encoder(images)
        outputs = classifier(features)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_acc = 100. * test_correct / test_total
print("\n" + "="*40)
print( "kết quả đánh giá cuối cùng" )
print(f"ĐỘ CHÍNH XÁC TRÊN TẬP TEST: {test_acc:.2f}%")
print("="*40)

NameError: name 'f_encoder' is not defined